In [1]:
import os
import json
import re
from pathlib import Path
from typing import List, Dict, Tuple, Optional
import pandas as pd
import pdfplumber
from tqdm.auto import tqdm

In [2]:
BASE_DIR = Path(r"D:\Final_GRAG")
GRI_STANDARDS_DIR = BASE_DIR / "GRI_standards"

METADATA_DIR = BASE_DIR / "metadata"
METADATA_DIR.mkdir(exist_ok=True)

UNITS_DIR = METADATA_DIR / "gri_units"

for dir_path in [UNITS_DIR]:
    dir_path.mkdir(exist_ok=True)

In [3]:
pdf_files = list(GRI_STANDARDS_DIR.glob("*.pdf"))
print(len(pdf_files))

38


## PDF parser

### Xử lý tên file 

In [4]:
def parse_gri_filename(filename):
    # Bỏ đuôi pdf
    name = filename.replace('.pdf', '')
    
    # Pattern: GRI {loại tiêu chuẩn}[_:] {tên tiêu chuẩn} {năm phát hành}
    pattern = r'GRI\s*(\d+)[_:]\s*([^0-9]+?)\s*(\d{4})'
    match = re.search(pattern, name)
    
    if match:
        standard_num = match.group(1)
        standard_name = match.group(2).strip().strip('_').strip()
        year = int(match.group(3))
        
        standard_id = f"GRI {standard_num}"
        
        # Chia các standards về các loại
        if standard_num in ['1', '2', '3']:
            standard_type = 'universal'
        elif standard_num in ['11', '12', '13', '14']:
            standard_type = 'sector'
        else:
            standard_type = 'topic'
        
        return {
            'standard_id': standard_id,
            'standard_num': standard_num,
            'standard_name': standard_name,
            'year': year,
            'standard_type': standard_type,
            'filename': filename
        }

### Xử lý text

In [5]:
def extract_pdf(pdf_path):
    with pdfplumber.open(pdf_path) as pdf:
        text = ""
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
    return text


def clean_text(text):
    # Loại bỏ khoảng trắng
    text = re.sub(r'\s+', ' ', text)
    # Loại bỏ các xuống dòng thừa
    text = re.sub(r'\n+', '\n', text)
    # Loại bỏ khoảng trắng đầu/cuối
    text = text.strip()
    return text

### Xử lý REQUIREMENTS

In [6]:
ROMAN_NUMERALS = {'i', 'ii', 'iii', 'iv', 'v', 'vi', 'vii', 'viii', 'ix', 'x', 'xi', 'xii', 'xiii', 'xiv', 'xv'}

ROMAN_PATTERN = r'(i{1,3}|iv|v|vi{1,3}|ix|x|xi{1,3}|xiv|xv)'

def parse_letter_marker(line):
    """
    Xử lý các req bậc 1 (a, b, c, ...) --> Trả về (chữ cái, phần còn lại của dòng).
    """
    match = re.match(r'^([a-z])[\.\)]\s+(.+)$', line, re.IGNORECASE)
    if match:
        letter = match.group(1).lower()
        # Bỏ qua số la mã
        if letter.lower() not in ROMAN_NUMERALS:
            return letter, match.group(2)
    return None, None

def parse_roman_marker(line):
    """
    Xử lý các req bậc 2 (i, ii, iii, ...) --> Trả về (số la mã, phần còn lại của dòng)
    """

    match = re.match(r'^(i{1,3}|iv|v|vi{1,3}|ix|x|xi{1,3}|xiv|xv)[\.\)]\s+(.+)$', line, re.IGNORECASE)
    if match:
        return match.group(1).lower(), match.group(2)
    return None, None

In [7]:
def extract_inline_sub_requirements(text):
    """
    (LỖI CHÍNH Ở BẢN TRƯỚC)
    Xử lý các sub-requirements inline: "... i. dòng 1; ii. dòng 2; iii. dòng 3; ..." --> Trả về (số la mã, sub-requirement-text).
    """
    sub_reqs = []
    pattern = rf'(?:^|[;:]\s*)({ROMAN_PATTERN})[\.\)]\s+([^;]+?)(?=;\s*{ROMAN_PATTERN}[\.\)]|;?\s*$|$)'
    
    matches = list(re.finditer(pattern, text, re.IGNORECASE))
    
    for match in matches:
        roman = match.group(1).lower()
        sub_text = match.group(2).strip().rstrip(';').strip()
        sub_reqs.append((roman, sub_text))
    
    return sub_reqs

def has_inline_sub_requirements(text):
    """
    Kiểm tra text có chứa sub-requirements inline không 
    """
    pattern = rf'[;:]\s*{ROMAN_PATTERN}[\.\)]\s+\w'
    return bool(re.search(pattern, text, re.IGNORECASE))


def split_text_at_sub_requirements(text):
    """
    Tách thành main text và sub-requirements.
    """
    # Tìm vị trí bắt đầu của sub-requirements
    pattern = rf'[;:]\s*{ROMAN_PATTERN}[\.\)]\s+'
    match = re.search(pattern, text, re.IGNORECASE)

    if not match:
        return text, []
    
    # Phần văn bản chính trước sub-requirements
    main_text = text[:match.start()].strip()

    # Phần văn bản chứa sub-requirements
    sub_req_text = text[match.start():]
    sub_reqs = extract_inline_sub_requirements(sub_req_text)
    
    return main_text, sub_reqs

In [8]:
def find_requirements_end(text, is_gri1 = False):
    """
    Xác định vị trí kết thúc của phần REQUIREMENTS.
    """
    if is_gri1:
        pattern = r'\n\s*(?:Guidance|GUIDANCE|Requirement\s+\d+:)'
    else:
        pattern = r'(?:\n\s*(?:GUIDANCE|Guidance)|(?:This disclosure|The (?:reporting )?organization is (?:expected|required) to|Recommendations?|Compilation requirements|Compiled GRI|^\d+\.))'
    
    match = re.search(pattern, text, re.IGNORECASE | re.MULTILINE)
    return match.start() if match else len(text)


def collect_continuation_lines(lines, start_idx):
    """
    Lấy dữ liệu cho đến khi gặp cản (chữ hoặc số la mã)
    """
    collected = []
    j = start_idx
    
    while j < len(lines):
        line = lines[j].strip()
        if not line:
            j += 1
            continue
        
        # Kiểm tra chữ cái tiếp theo
        letter, _ = parse_letter_marker(line)
        if letter:
            break
        
        # Kiểm tra số la mã tiếp theo
        roman, _ = parse_roman_marker(line)
        if roman:
            break
        
        collected.append(line)
        j += 1
    
    return collected, j


def create_requirement_dict(disclosure_id, disclosure_name, req_id, req_text, hierarchy_level, parent_req):
    return {
        'disclosure_id': disclosure_id,
        'disclosure_name': disclosure_name,
        'requirement_id': req_id,
        'requirement_text': req_text,
        'hierarchy_level': hierarchy_level,
        'parent_requirement': parent_req
    }

In [9]:
def _normalize_parent_prefix(text: str) -> str:
    """Bảo đảm prefix kết thúc bằng ':' để khi ghép với child text đọc tự nhiên."""
    s = text.strip().rstrip(';,. ')
    if not s.endswith(':'):
        s = s + ':'
    return s


def save_requirement_with_sub_reqs(requirements, disclosure_id, disclosure_name, letter, letter_text, line_based_sub_reqs):
    """
    Xử lý req bậc 1 và bậc 2 (cả line-based và inline sub-requirements).
    Khi có sub-requirements: gộp parent text làm prefix vào từng child text,
    KHÔNG tạo record riêng cho parent level-1 (tránh nhiễu RAG).
    """
    full_text = ' '.join(letter_text).strip()
    
    # Kiểm tra sub-requirements inline (i., ii., etc. trong cùng một dòng)
    inline_sub_reqs = []
    main_text = full_text
    
    if has_inline_sub_requirements(full_text):
        main_text, inline_sub_reqs = split_text_at_sub_requirements(full_text)
        main_text = main_text.rstrip(':').strip()
    
    # Tránh duplicate line-based và inline sub-requirements
    all_sub_reqs = line_based_sub_reqs if line_based_sub_reqs else inline_sub_reqs
    
    if all_sub_reqs:
        # Gộp parent text vào mỗi child: "Base year...: the rationale for choosing it;"
        prefix = _normalize_parent_prefix(main_text)
        for roman, sub_text in all_sub_reqs:
            sub_text = sub_text.strip()
            if len(sub_text) >= 5:
                merged_text = f"{prefix} {sub_text}"
                requirements.append(create_requirement_dict(
                    disclosure_id, disclosure_name,
                    f"{letter}.{roman}",
                    merged_text, 2, letter
                ))
    else:
        # Không có sub-requirements --> thêm main requirement ở level-1
        if len(main_text) >= 10:
            requirements.append(create_requirement_dict(
                disclosure_id, disclosure_name, letter, main_text, 1, None
            ))

def parse_requirements_from_section(req_section, disclosure_id, disclosure_name):
    """
    Xử lý requirements từ phần REQUIREMENTS của các disclosure.
    """
    requirements = []
    lines = req_section.split('\n')
    
    current_letter = None
    current_letter_text = []
    line_based_sub_reqs = []
    
    i = 0
    while i < len(lines):
        line = lines[i].strip()
        if not line:
            i += 1
            continue
        
        letter, letter_content = parse_letter_marker(line)
        
        roman, roman_content = parse_roman_marker(line)

        # Xử lý cho requirement bậc 1 (chữ cái)
        if letter:
            if current_letter:
                save_requirement_with_sub_reqs(
                    requirements, disclosure_id, disclosure_name,
                    current_letter, current_letter_text, 
                    line_based_sub_reqs
                )
            current_letter = letter
            current_letter_text = [letter_content]
            line_based_sub_reqs = []

        # Xử lý cho requirement bậc 2 (số la mã)    
        elif roman:
            sub_text = [roman_content]
            continuation, new_idx = collect_continuation_lines(lines, i + 1)
            sub_text.extend(continuation)
            i = new_idx - 1
            
            full_sub_text = ' '.join(sub_text).strip()
            if len(full_sub_text) >= 5 and current_letter:
                line_based_sub_reqs.append((roman, full_sub_text))
        else:
            if current_letter:
                current_letter_text.append(line)
        
        i += 1
    
    # Save requirement vừa xử lý
    if current_letter:
        save_requirement_with_sub_reqs(
            requirements, disclosure_id, disclosure_name,
            current_letter, current_letter_text,
            line_based_sub_reqs
        )
    
    return requirements

In [10]:
def extract_requirements_gri1(text):
    """
    Sử dụng "Requirement X: Title" format thay vì "Disclosure X-Y".
    """
    requirements = []
    
    # Pattern: "Requirement X: Title"
    disclosure_pattern = r'Requirement\s+(\d+):\s+([^\n]+)'
    
    matches = list(re.finditer(disclosure_pattern, text, re.IGNORECASE))
    
    for i, match in enumerate(matches):
        req_num = match.group(1).strip()
        req_name = re.sub(r'\s+', ' ', match.group(2).strip())
        
        # Trích xuất text cho requirement 
        start_pos = match.end()
        end_pos = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        section_text = text[start_pos:end_pos]
        
        # Tìm chữ cái bắt đầu (a.)
        req_start = re.search(r'\s*a\.', section_text)
        if not req_start:
            continue
        
        # Tìm kết thúc phần requirements (trước GUIDANCE)
        section_end = find_requirements_end(section_text[req_start.start():], is_gri1=True)
        req_section = section_text[req_start.start():req_start.start() + section_end]
        
        # Trích xuất requirements
        parsed = parse_requirements_from_section(req_section, req_num, req_name)
        requirements.extend(parsed)
    
    return requirements

def extract_requirements_standard(text):
    """
    Sử dụng "Disclosure X-Y" format để xử lý các GRI standards còn lại.
    """
    requirements = []
    
    # Pattern: "Disclosure X-Y Title"
    disclosure_pattern = r'Disclosure\s+(\d+-\d+)\s+([^\n]+?)(?:\n|$)'
    
    matches = list(re.finditer(disclosure_pattern, text, re.IGNORECASE))
    
    for i, match in enumerate(matches):
        disclosure_id = match.group(1).strip()
        disclosure_name = re.sub(r'\s+', ' ', match.group(2).strip())
        
        # Trích xuất text cho requirement 
        start_pos = match.end()
        end_pos = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        disclosure_text = text[start_pos:end_pos]
        
        # Tìm phần REQUIREMENTS
        req_start = re.search(
            r'(?:The\s+(?:reporting\s+)?organization\s+shall[:\s]+)?REQUIREMENTS',
            disclosure_text, re.IGNORECASE
        )
        if not req_start:
            continue
        
        # Trích xuất text cho requirement 
        req_text_start = req_start.end()
        section_end = find_requirements_end(disclosure_text[req_text_start:], is_gri1=False)
        req_section = disclosure_text[req_text_start:req_text_start + section_end]
        
        # Trích xuất requirements
        parsed = parse_requirements_from_section(req_section, disclosure_id, disclosure_name)
        requirements.extend(parsed)
    
    return requirements

In [11]:
def extract_requirements(text, standard):
    """
    Trích xuất requirements từ GRI standards.
    """
    is_gri1 = standard.get('standard_num') == '1'
    
    if is_gri1:
        requirements = extract_requirements_gri1(text)
    else:
        requirements = extract_requirements_standard(text)
    
    # is_omittable = True cho tất cả requirements, ngoại trừ một số disclosures bắt buộc
    non_omittable_disclosures = {"2-1", "2-2", "2-3", "2-4", "2-5", "3-1", "3-2"}
    
    for req in requirements:
        req['requirement_type'] = 'shall'
        req['is_omittable'] = req.get('disclosure_id') not in non_omittable_disclosures
    
    return requirements

## Xử lý toàn bộ GRI standards

In [12]:
all_units = []

for pdf_path in tqdm(pdf_files, desc="Processing PDFs"):
    standard_info = parse_gri_filename(pdf_path.name)
    text = extract_pdf(pdf_path)

    if not text:
        continue

    requirements = extract_requirements(text, standard_info)

    if not requirements:
        continue

    for req in requirements:
        req_id_clean = req["requirement_id"].replace(".", "-")
        base_unit_id = (
            f"{standard_info['standard_id'].replace(' ', '')}-"
            f"{standard_info['year']}-"
            f"D{req['disclosure_id']}-"
            f"R{req_id_clean}"
        )

        unit_id = base_unit_id
        unit = {
            "unit_id": unit_id,
            "standard_id": standard_info["standard_id"],
            "standard_name": standard_info["standard_name"],
            "standard_type": standard_info["standard_type"],
            "disclosure_id": req["disclosure_id"],
            "disclosure_name": req["disclosure_name"],
            "requirement_id": req["requirement_id"],
            "requirement_text": req["requirement_text"][:8192],
            "requirement_type": req["requirement_type"],
            "is_omittable": req["is_omittable"],
            "year": standard_info["year"],
            "hierarchy_level": req.get("hierarchy_level", 1),
            "parent_requirement": req.get("parent_requirement", None),
        }

        all_units.append(unit)


Processing PDFs:   0%|          | 0/38 [00:00<?, ?it/s]

In [13]:
all_units_df = pd.DataFrame(all_units)
all_units_df[:5]

,unit_id,standard_id,standard_name,standard_type,disclosure_id,disclosure_name,requirement_id,requirement_text,requirement_type,is_omittable,year,hierarchy_level,parent_requirement
0,GRI101-2024-D101-1-Ra,GRI 101,Biodiversity,topic,101-1,Policies to halt and reverse,a,describe its policies or commitments to halt a...,shall,True,2024,1,None
1,GRI101-2024-D101-1-Rb,GRI 101,Biodiversity,topic,101-1,Policies to halt and reverse,b,report the extent to which these policies or c...,shall,True,2024,1,None
2,GRI101-2024-D101-1-Rc,GRI 101,Biodiversity,topic,101-1,Policies to halt and reverse,c,report the goals and targets to halt and rever...,shall,True,2024,1,None
3,GRI101-2024-D101-2-Ra-i,GRI 101,Biodiversity,topic,101-2,Management of biodiversity impacts,a.i,report how it applies the mitigation hierarchy...,shall,True,2024,2,a
4,GRI101-2024-D101-2-Ra-ii,GRI 101,Biodiversity,topic,101-2,Management of biodiversity impacts,a.ii,report how it applies the mitigation hierarchy...,shall,True,2024,2,a


## Lưu metadata

In [14]:
def sanitize_for_json(obj):
    import numpy as np
    if isinstance(obj, dict):
        return {k: sanitize_for_json(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [sanitize_for_json(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return sanitize_for_json(obj.tolist())
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    return obj

units_file = UNITS_DIR / "gri_units.json"
with open(units_file, 'w', encoding='utf-8') as f:
    json.dump(sanitize_for_json(all_units), f, indent=2, ensure_ascii=False)

In [15]:
# Save all_units as csv for easier viewing
units_csv_file = UNITS_DIR / "gri_units.csv"
all_units_df.to_csv(units_csv_file, index=False)

In [19]:
n_rows = len(all_units_df)
print(f"Số PDF: {len(pdf_files)}")
print(f"Số dòng requirements: {n_rows}")

Số PDF: 38
Số dòng requirements: 787
